# Track E — Cross-Model CAI Study

**Central question**: *Is CAI a property of the template alone, or of the template-model pairing?*

If templates generalize across models → CAI is a template quality metric that can be published as a property of the template.  
If they don't → CAI must always be labeled with the model it was measured on.

## Design
- **6 templates** × **3+ models** × **3 runs** each = 54–72 LLM calls
- Estimated cost: ~$2–5 depending on models used
- Results cached to `cai_cross_model_results.csv`

## Models tested
- `gpt-4o-mini` (default, cheap, fast)
- `gpt-4o` (stronger)
- `claude-3-5-haiku-20241022` (Anthropic, if key available)

## Templates tested
A mix of free cognitive templates from different domains.

## Key metrics
- CAI mean and std per template × model
- Coefficient of variation (CV) = std/mean → stability score
- Generalization index: does rank order of templates hold across models?
- Kendall's τ between model pairs for template CAI ranking

In [ ]:
import os, sys, json, pathlib, random, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
matplotlib.rcParams["figure.dpi"] = 120
from scipy import stats
from dotenv import load_dotenv

load_dotenv(pathlib.Path("../../.env"))
sys.path.insert(0, str(pathlib.Path("../../src")))

from mycontext import Context
from mycontext.intelligence.context_amplification import ContextAmplificationIndex

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ── Configuration ─────────────────────────────────────────────────────────────
N_RUNS = 3          # runs per template × model pair
CACHE_FILE = pathlib.Path("cai_cross_model_results.csv")
FORCE_RERUN = False  # set True to overwrite cache

# Models to test — comment out any you don't have API keys for
MODELS = [
    {"provider": "openai",    "model": "gpt-4o-mini",               "label": "GPT-4o-mini"},
    {"provider": "openai",    "model": "gpt-4o",                    "label": "GPT-4o"},
    # {"provider": "anthropic", "model": "claude-3-5-haiku-20241022", "label": "Claude-3.5-Haiku"},
]

print(f"Models: {[m['label'] for m in MODELS]}")
print(f"Runs per pair: {N_RUNS}")
print(f"Total LLM calls (est): {len(MODELS) * 6 * N_RUNS * 2} (2 calls per CAI measurement)")  

In [ ]:
# ── 6 templates × 6 representative questions ──────────────────────────────────

TEMPLATE_QUESTIONS = [
    {
        "template": "root_cause_analyzer",
        "question": "Why did our API response times triple after the v2.4 deployment last Tuesday?",
        "domain": "Engineering",
    },
    {
        "template": "scenario_planner",
        "question": "What are the key scenarios we should plan for as AI regulation evolves over the next 3 years?",
        "domain": "Strategy",
    },
    {
        "template": "risk_assessor",
        "question": "What are the main risks of migrating our monolithic app to microservices architecture?",
        "domain": "Technology",
    },
    {
        "template": "hypothesis_generator",
        "question": "Why has our user retention dropped 15% in the past quarter despite no product changes?",
        "domain": "Product",
    },
    {
        "template": "data_analyzer",
        "question": "Our Q3 revenue grew 8% but gross margin dropped 4 points — what could explain this pattern?",
        "domain": "Finance",
    },
    {
        "template": "synthesis_builder",
        "question": "Synthesize the key lessons from the three failed product launches in our company's history.",
        "domain": "Strategy",
    },
]

print(f"Templates: {len(TEMPLATE_QUESTIONS)}")
for tq in TEMPLATE_QUESTIONS:
    print(f"  {tq['template']} [{tq['domain']}]")

In [ ]:
# ── Run CAI measurements (cached) ─────────────────────────────────────────────

def run_all_measurements():
    rows = []
    total = len(MODELS) * len(TEMPLATE_QUESTIONS) * N_RUNS
    done = 0

    for model_cfg in MODELS:
        provider = model_cfg["provider"]
        model    = model_cfg["model"]
        label    = model_cfg["label"]
        cai = ContextAmplificationIndex(provider=provider, eval_mode="heuristic", model=model)

        for tq in TEMPLATE_QUESTIONS:
            for run in range(N_RUNS):
                done += 1
                print(f"  [{done:3d}/{total}] {label} | {tq['template']} | run {run+1}")
                try:
                    result = cai.measure(
                        question=tq["question"],
                        template_name=tq["template"],
                        baseline="minimal",  # will use "raw" if minimal not supported yet
                    )
                    rows.append({
                        "model_label":   label,
                        "provider":      provider,
                        "model":         model,
                        "template":      tq["template"],
                        "domain":        tq["domain"],
                        "run":           run + 1,
                        "cai_overall":   result.cai_overall,
                        "raw_score":     result.raw_score.overall,
                        "template_score": result.templated_score.overall,
                        "verdict":       result.verdict,
                        **{f"cai_{d.value}": result.cai_dimensions.get(d, 0) for d in result.cai_dimensions},
                    })
                    time.sleep(0.5)  # rate limiting
                except Exception as e:
                    print(f"    ERROR: {e}")
                    rows.append({
                        "model_label": label, "provider": provider, "model": model,
                        "template": tq["template"], "domain": tq["domain"],
                        "run": run + 1, "cai_overall": None, "error": str(e),
                    })
    return pd.DataFrame(rows)


if CACHE_FILE.exists() and not FORCE_RERUN:
    results_df = pd.read_csv(CACHE_FILE)
    print(f"Loaded from cache: {len(results_df)} rows")
else:
    print(f"Running {len(MODELS) * len(TEMPLATE_QUESTIONS) * N_RUNS} CAI measurements...")
    results_df = run_all_measurements()
    results_df.to_csv(CACHE_FILE, index=False)
    print(f"\nSaved to {CACHE_FILE}")

valid = results_df[results_df["cai_overall"].notna()]
print(f"\nValid measurements: {len(valid)} / {len(results_df)}")
print(valid.groupby(["model_label", "template"])["cai_overall"].mean().round(3).unstack())

In [ ]:
# ── Reliability analysis: mean ± std, coefficient of variation ───────────────

valid = results_df[results_df["cai_overall"].notna()].copy()

reliability = valid.groupby(["model_label","template"])["cai_overall"].agg(
    mean="mean", std="std", count="count"
).reset_index()
reliability["cv"] = reliability["std"] / reliability["mean"].clip(lower=0.01)
reliability["stability"] = reliability["cv"].apply(
    lambda cv: "stable" if cv < 0.15 else ("moderate" if cv < 0.30 else "unstable")
)

print("Reliability summary (mean ± std, CV, stability):")
print(reliability.to_string(index=False))

# Overall stability per template (across all models)
print("\nTemplate stability (averaged across models):")
tpl_stability = reliability.groupby("template")[["mean","std","cv"]].mean().round(3)
tpl_stability["stability"] = tpl_stability["cv"].apply(
    lambda cv: "✅ stable" if cv < 0.15 else ("⚠️ moderate" if cv < 0.30 else "❌ unstable")
)
print(tpl_stability.to_string())

In [ ]:
# ── Visualizations ───────────────────────────────────────────────────────────

pivot_mean = reliability.pivot(index="template", columns="model_label", values="mean")
pivot_cv   = reliability.pivot(index="template", columns="model_label", values="cv")

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plot 1: CAI heatmap (mean)
ax = axes[0]
sns.heatmap(pivot_mean, annot=True, fmt=".2f", cmap="YlOrRd",
            vmin=0.8, vmax=2.5, ax=ax, linewidths=0.5, cbar_kws={"label": "CAI (mean)"})
ax.set_title("CAI Mean per Template × Model", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# Plot 2: CV heatmap (stability)
ax = axes[1]
sns.heatmap(pivot_cv, annot=True, fmt=".2f", cmap="RdYlGn_r",
            vmin=0, vmax=0.4, ax=ax, linewidths=0.5, cbar_kws={"label": "CV (lower=more stable)"})
ax.set_title("Stability (Coefficient of Variation)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

# Plot 3: Template rank consistency across models
ax = axes[2]
models_available = list(pivot_mean.columns)
x = np.arange(len(pivot_mean.index))
width = 0.8 / max(len(models_available), 1)
colors_m = plt.cm.Set2(np.linspace(0, 0.8, len(models_available)))

for i, model_label in enumerate(models_available):
    vals = pivot_mean[model_label].values
    ax.bar(x + i * width - (len(models_available)-1)*width/2, vals,
           width=width, label=model_label, color=colors_m[i], alpha=0.85)

ax.axhline(1.0, color="black", linestyle="--", linewidth=0.8, label="Neutral (1.0x)")
ax.set_xticks(x)
ax.set_xticklabels(pivot_mean.index, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("CAI (mean across runs)")
ax.set_title("CAI per Template — Model Comparison", fontweight="bold")
ax.legend(fontsize=9)

plt.suptitle("Cross-Model CAI Study: Template Lift Generalization", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../cai_cross_model.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: research/cai_cross_model.png")

In [ ]:
# ── Generalization index: do templates rank the same way across models? ────────

models_available = list(pivot_mean.columns)
if len(models_available) >= 2:
    print("Kendall's τ for template rank ordering across model pairs:")
    print("(τ close to 1.0 = templates rank identically across models)\n")

    for i in range(len(models_available)):
        for j in range(i+1, len(models_available)):
            m1, m2 = models_available[i], models_available[j]
            rank1 = pivot_mean[m1].rank()
            rank2 = pivot_mean[m2].rank()
            tau, p = stats.kendalltau(rank1, rank2)
            label = "✅ generalizes" if tau > 0.7 else ("⚠️ partial" if tau > 0.4 else "❌ model-specific")
            print(f"  {m1} vs {m2}: τ={tau:.3f} (p={p:.4f})  {label}")

    print("\nConclusion:")
    avg_tau = np.mean([
        stats.kendalltau(pivot_mean[m1].rank(), pivot_mean[m2].rank())[0]
        for i, m1 in enumerate(models_available)
        for m2 in models_available[i+1:]
    ])
    if avg_tau > 0.7:
        print(f"  Average τ = {avg_tau:.3f} → Templates generalize across models.")
        print("  CAI can be treated as a property of the template, not the template-model pair.")
    elif avg_tau > 0.4:
        print(f"  Average τ = {avg_tau:.3f} → Partial generalization.")
        print("  CAI is moderately model-dependent. Report with model label.")
    else:
        print(f"  Average τ = {avg_tau:.3f} → Poor generalization.")
        print("  CAI is model-specific. Always report which model was used.")
else:
    print("Only one model tested — run with multiple models to compute generalization index.")

## Results Interpretation

### What we're looking for

| Result | Interpretation | Implication |
|--------|---------------|-------------|
| Kendall τ > 0.70 across all model pairs | Templates generalize | CAI is a template property — can be published as a stable metric |
| τ 0.40–0.70 | Partial generalization | Report CAI with model label; use multi-model average |
| τ < 0.40 | Model-specific | CAI measures template-model fit, not template quality alone |

### CV / stability thresholds

| CV | Stability | Meaning |
|----|-----------|---------|
| < 0.15 | Stable | 3 runs is enough; the template reliably produces this lift |
| 0.15–0.30 | Moderate | Run 5+ times; report CI |
| > 0.30 | Unstable | Template lift is noise-dominated; likely model sensitivity |

### What to do with the findings
- **If generalizes**: implement `measure_reliable(n_runs=3)` with minimal baseline and ship
- **If model-specific**: add `model` parameter to CAI report, run once per model in the benchmark
- **High CV templates**: investigate what makes them noisy — often short questions with many valid response structures